# Stage 15: Orchestration & System Design

Decomposing the hobby-box EV pipeline (today: `project/notebooks/project_pipeline.ipynb`, run by hand) into schedulable tasks. Project deliverables: `project/docs/orchestration_plan.md`, `project/src/run_step.py`.

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas

## 1) Project task decomposition

7 tasks. Every task reads named inputs and overwrites named outputs (no appends, no in-place edits) — so every task is idempotent and safe to re-run after a partial failure.

In [2]:
import pandas as pd
pd.set_option("display.max_colwidth", 60)

tasks = pd.DataFrame([
    ("T1 build_raw",   "src/build_raw_dataset.py",   "transcription constants (in script)", "data/raw/card_tiers.csv, box_products.csv, tier_comps.csv, chase_cards.csv", True),
    ("T2 features",    "src/features.py + src/outliers.py", "data/raw/card_tiers.csv, box_products.csv", "data/processed/features.csv", True),
    ("T3 train_model", "src/model.py",              "data/raw/card_tiers.csv, box_products.csv", "model/model.pkl", True),
    ("T4 ev_report",   "src/run_step.py -> src/ev.py", "data/raw/*.csv",                 "data/processed/ev_report.csv", True),
    ("T5 evaluate",    "src/evaluation.py",         "data/raw/card_tiers.csv",           "data/processed/scenario_results.csv", True),
    ("T6 charts",      "src/plotting.py",           "data/processed/ev_report.csv",      "reports/images/ev_vs_price.png, ev_per_dollar.png, sensitivity_values.png", True),
    ("T7 monitor",     "Stage 14 baseline cell",    "data/raw/card_tiers.csv, data/processed/ev_report.csv, model/model.pkl", "data/processed/monitoring_baselines.json", True),
], columns=["task", "function", "inputs", "outputs", "idempotent"])
tasks

,task,function,inputs,outputs,idempotent
0,T1 build_raw,src/build_raw_dataset.py,transcription constants (in script),"data/raw/card_tiers.csv, box_products.csv, tier_comps.cs...",True
1,T2 features,src/features.py + src/outliers.py,"data/raw/card_tiers.csv, box_products.csv",data/processed/features.csv,True
2,T3 train_model,src/model.py,"data/raw/card_tiers.csv, box_products.csv",model/model.pkl,True
3,T4 ev_report,src/run_step.py -> src/ev.py,data/raw/*.csv,data/processed/ev_report.csv,True
4,T5 evaluate,src/evaluation.py,data/raw/card_tiers.csv,data/processed/scenario_results.csv,True
5,T6 charts,src/plotting.py,data/processed/ev_report.csv,"reports/images/ev_vs_price.png, ev_per_dollar.png, sensi...",True
6,T7 monitor,Stage 14 baseline cell,"data/raw/card_tiers.csv, data/processed/ev_report.csv, m...",data/processed/monitoring_baselines.json,True


## 2) Dependencies (DAG)

```
T1 build_raw
 ├─> T2 features ──> T3 train_model ─┐
 ├─> T4 ev_report ──────────────────┼─> T6 charts
 ├─> T5 evaluate                    └─> T7 monitor
 └─> (T7 also needs T3)
```

`T2`, `T4`, `T5` each read only `data/raw/*.csv`, so they **run in parallel** after `T1`. Critical path: `T1 → T2 → T3 → T7`.

In [3]:
dag = {
    "T1_build_raw":   [],
    "T2_features":    ["T1_build_raw"],
    "T3_train_model": ["T2_features"],
    "T4_ev_report":   ["T1_build_raw"],
    "T5_evaluate":    ["T1_build_raw"],
    "T6_charts":      ["T4_ev_report"],
    "T7_monitor":     ["T3_train_model", "T4_ev_report"],
}
parallel_after_T1 = [k for k, deps in dag.items() if deps == ["T1_build_raw"]]
print("can run in parallel after T1:", parallel_after_T1)
dag

can run in parallel after T1: ['T2_features', 'T4_ev_report', 'T5_evaluate']


{'T1_build_raw': [],
 'T2_features': ['T1_build_raw'],
 'T3_train_model': ['T2_features'],
 'T4_ev_report': ['T1_build_raw'],
 'T5_evaluate': ['T1_build_raw'],
 'T6_charts': ['T4_ev_report'],
 'T7_monitor': ['T3_train_model', 'T4_ev_report']}

## 3) Logging & checkpoints

- **Logging:** `logging` at INFO to stderr, two lines per task — `start` (params) and `end` (rows in/out, artifact path, seconds). A scheduled run appends stderr to `logs/pipeline_YYYY-MM-DD.log`.
- **Checkpoints:** each task's output file *is* its checkpoint (CSV / JSON / PKL / PNG, all < 1 MB, seconds to rebuild). On failure, restart from the first task whose output is missing or older than its inputs. No separate state store — the pipeline is too small to justify one.

In [4]:
logging_plan = pd.DataFrame([
    ("T1 build_raw",   "start; rows written per CSV; end + seconds",       "data/raw/*.csv"),
    ("T2 features",    "start; rows in / rows out; NaN counts per feature","data/processed/features.csv"),
    ("T3 train_model", "start; n_train/n_test; holdout R2, RMSE; end",     "model/model.pkl"),
    ("T4 ev_report",   "start (base_card_value); rows, positive_ev; seconds","data/processed/ev_report.csv"),
    ("T5 evaluate",    "start; slope per scenario; bootstrap CI",          "data/processed/scenario_results.csv"),
    ("T6 charts",      "one line per figure saved",                        "reports/images/*.png"),
    ("T7 monitor",     "each baseline metric + its alert threshold",       "data/processed/monitoring_baselines.json"),
], columns=["task", "log_messages", "checkpoint_artifact"])
logging_plan

,task,log_messages,checkpoint_artifact
0,T1 build_raw,start; rows written per CSV; end + seconds,data/raw/*.csv
1,T2 features,start; rows in / rows out; NaN counts per feature,data/processed/features.csv
2,T3 train_model,"start; n_train/n_test; holdout R2, RMSE; end",model/model.pkl
3,T4 ev_report,"start (base_card_value); rows, positive_ev; seconds",data/processed/ev_report.csv
4,T5 evaluate,start; slope per scenario; bootstrap CI,data/processed/scenario_results.csv
5,T6 charts,one line per figure saved,reports/images/*.png
6,T7 monitor,each baseline metric + its alert threshold,data/processed/monitoring_baselines.json


## 4) Right-sizing automation

**Automate now** (weekly `cron` job chaining `python src/run_step.py <task>` calls): T4 ev_report, T6 charts, T7 monitor. They are pure functions of committed data, need no judgement, and are exactly what the Stage 14 monitoring watches. `src/run_step.py` is step one — the `ev_report` task already runs from the CLI with logging + retry.

**Automate later:** T2 features, T3 train_model — mechanical, but only worth scheduling once real eBay comps replace the `rough_estimate_v0` placeholder, because retraining on the placeholder ladder doesn't change anything.

**Keep manual:** T1 build_raw — the raw CSVs are hand-transcribed from Topps / checklistinsider / blowoutcards odds sheets. A person has to read them and edit `build_raw_dataset.py`. Automated scraping is out of scope and against several sites' ToS.

**Retry policy:** retry only steps that fail *transiently* (I/O) — T4 gets 3 attempts with linear backoff. Steps that fail on bad data (T2/T3/T5) stop the run and page the owner; retrying wouldn't help.

Scope: no Airflow / Prefect. A `Makefile` + `cron`, total runtime < 5 min, is right-sized for a single-analyst project.

## 5) Refactor one task into a function + CLI

Done in the project repo as `project/src/run_step.py`. The core is below (`ev_report`), plus the retry wrapper it uses. Run it with:

```
cd project && python src/run_step.py ev_report -v
```

In [5]:
import argparse, logging, sys, time
from pathlib import Path

log = logging.getLogger("run_step")

def ev_report(out_path: str, base_card_value: float = 0.20) -> Path:
    """Recompute the EV table (src/ev.py::ev_table) and write it to out_path. Idempotent."""
    from src.ev import ev_table
    t0 = time.perf_counter()
    log.info("ev_report: start  base_card_value=%.2f", base_card_value)
    df = ev_table(base_card_value=base_card_value)
    out = Path(out_path); out.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out, index=False)
    log.info("ev_report: wrote %s  rows=%d  positive_ev=%d  (%.2fs)",
             out, len(df), int((df["ev_per_price"] >= 1).sum()), time.perf_counter() - t0)
    return out

def main(argv=None) -> int:
    p = argparse.ArgumentParser(description="Run one hobby-box EV pipeline task.")
    p.add_argument("step", choices=["ev_report"])
    p.add_argument("--out", default="data/processed/ev_report.csv")
    p.add_argument("--base-card-value", type=float, default=0.20)
    p.add_argument("-v", "--verbose", action="store_true")
    a = p.parse_args(argv)
    logging.basicConfig(level=logging.DEBUG if a.verbose else logging.INFO,
                        format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    try:
        retry(ev_report, out_path=a.out, base_card_value=a.base_card_value)
    except Exception:
        return 1
    return 0

# (in the repo this file ends with:  if __name__ == "__main__": sys.exit(main()))
print("see project/src/run_step.py for the full CLI wrapper")

see project/src/run_step.py for the full CLI wrapper


### Retry wrapper (linear backoff)

In [6]:
import time

def retry(fn, *args, tries: int = 3, backoff: float = 1.0, **kwargs):
    """Call fn with linear backoff; re-raise the last error after `tries` attempts."""
    for attempt in range(1, tries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            if attempt == tries:
                logging.getLogger("run_step").error(
                    "%s failed after %d attempts: %s", getattr(fn, "__name__", fn), tries, e)
                raise
            wait = backoff * attempt
            logging.getLogger("run_step").warning(
                "%s attempt %d/%d failed (%s); retrying in %.1fs",
                getattr(fn, "__name__", fn), attempt, tries, e, wait)
            time.sleep(wait)

# demo: a task that fails twice then succeeds
_calls = {"n": 0}
def flaky():
    _calls["n"] += 1
    if _calls["n"] < 3:
        raise IOError(f"transient failure #{_calls['n']}")
    return "ok"

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
print("retry(flaky) ->", retry(flaky, tries=3, backoff=0.05))

WARNING run_step: flaky attempt 1/3 failed (transient failure #1); retrying in 0.1s
WARNING run_step: flaky attempt 2/3 failed (transient failure #2); retrying in 0.1s


retry(flaky) -> ok
